In [1]:
import pandas as pd
import numpy as np
import uncertainties as uc

pd.options.display.float_format ='{:,.1e}'.format
import matplotlib.pyplot as plt
%matplotlib inline  
from IPython.display import Image
from IPython.core.display import HTML 

from openpyxl import load_workbook

from scipy import integrate
import scipy.stats as stats
from uncertainties import ufloat
from uncertainties import unumpy
from itertools import groupby

from scipy.stats import gmean

%run Utility_Functions.ipynb

# Result summary
This notebook able to run all other notebooks and gather all the results in one place

## Running all notebooks 

Creating a class that can load a notebook and execute it

In [2]:
import io, os, sys, types
from IPython import get_ipython
from nbformat import read
from IPython.core.interactiveshell import InteractiveShell

def find_notebook(fullname, path=None):
    """find a notebook, given its fully qualified name and an optional path

    This turns "foo.bar" into "foo/bar.ipynb"
    and tries turning "Foo_Bar" into "Foo Bar" if Foo_Bar
    does not exist.
    """
    name = fullname.rsplit('.', 1)[-1]
    if not path:
        path = ['']
    for d in path:
        nb_path = os.path.join(d, name + ".ipynb")
        if os.path.isfile(nb_path):
            return nb_path
        # let import Notebook_Name find "Notebook Name.ipynb"
        nb_path = nb_path.replace("_", " ")
        if os.path.isfile(nb_path):
            return nb_path



class NotebookLoader(object):
    """Module Loader for Jupyter Notebooks"""
    def __init__(self, path=None):
        self.shell = InteractiveShell.instance()
        self.path = path

    def load_module(self, fullname):
        """import a notebook as a module"""
        path = find_notebook(fullname, self.path)

        print ("importing Jupyter notebook from %s" % path)

        # load the notebook object
        with io.open(path, 'r', encoding='utf-8') as f:
            nb = read(f, 4)


        # create the module and add it to sys.modules
        # if name in sys.modules:
        #    return sys.modules[name]
        mod = types.ModuleType(fullname)
        mod.__file__ = path
        mod.__loader__ = self
        mod.__dict__['get_ipython'] = get_ipython
        sys.modules[fullname] = mod

        # extra work to ensure that magics that would affect the user_ns
        # actually affect the notebook module's ns
        save_user_ns = self.shell.user_ns
        self.shell.user_ns = mod.__dict__

        try:
            for cell in nb.cells:
                if cell.cell_type == 'code':
                    # transform the input to executable Python
                    code = self.shell.input_transformer_manager.transform_cell(cell.source)
                    exec(code, mod.__dict__);
        finally:
            self.shell.user_ns = save_user_ns
        return mod

In [7]:
%%capture  

#suppres the many outputs of the notebook for this run  
 
notebooks = ['Erythrocytes','Neutrophils','Lymphocytes','Monocytes','GIT Epithelial Cells', 'Skin cells',
             'Endothelial cells','Lung cells', 'Adipocytes','Myocytes','Cardiomyocytes','Brain Cells', 'Liver cells']
NBL = NotebookLoader()
for nb_name in notebooks:
    NBL.load_module(nb_name)


KeyError: '% T cells'

## Loading the concentated data
Importing from the excel into a CellTypesResDF object and using it to print all the results.
In addition, the object method enable to deal with the data. For example the extended Dataframe of all the results including the upper and lower bound is computed

In [8]:
cell_types_data = pd.read_excel('Summary.xlsx','Colors', index_col=0, usecols=range(4))
cell_types_data = cell_types_data.drop('Other')

cells = CellTypesResDF(cell_types_data.index,un_type='mixed')
cells.import_celltypes_from_xlsx('Summary.xlsx')
cells.print_params()
cells.get_extended_dataframe()

Erythrocytes:
Number of Erythrocytes is: (2.5±0.1)×10¹³ cells
Lifespan of Erythrocytes is: 119±3 days
Cellular turnover rate of Erythrocytes is: (2.1±0.1)×10¹¹ cells per day
Cell mass of Erythrocytes is: 94±3 pg
Cellular mass turnover rate of Erythrocytes is: 20.00±1.00 grams per day
Total cellular mass of Erythrocytes is: 2380±100 grams
______________________________
Neutrophils:
Number of Neutrophils is: (6.4±0.6)×10¹¹ cells
Lifespan of Neutrophils is: 6.6±0.5 days
Cellular turnover rate of Neutrophils is: (6.0±1.5)×10¹⁰ cells per day
Cell mass of Neutrophils is: 300±30 pg
Cellular mass turnover rate of Neutrophils is: 18±5 grams per day
Total cellular mass of Neutrophils is: 190±30 grams
______________________________
Monocytes:
Number of Monocytes is: 5e+09 cells (SD range: 4e+09 - 8e+09 cells)
Lifespan of Monocytes is: 4 days (SD range: 3 - 4 days)
Cellular turnover rate of Monocytes is: 2e+09 cells per day (SD range: 1e+09 - 2e+09 cells per day)
Cell mass of Monocytes is: 460 pg 

/var/folders/75/2z9jf7bx0sqf07ypmf4yqgyhwmbx65/T/ipykernel_39940/1362151784.py:19: RuntimeWarning: invalid value encountered in log10
  log_vals = np.log10(values.astype(float))
/var/folders/75/2z9jf7bx0sqf07ypmf4yqgyhwmbx65/T/ipykernel_39940/1362151784.py:21: RuntimeWarning: invalid value encountered in remainder
  another_dig = 1*(log_vals %1 < 0.23)


,number,number lower bound,number upper bound,lifespan,lifespan lower bound,lifespan upper bound,cellular turnover rate,cellular turnover rate lower bound,cellular turnover rate upper bound,cell mass,...,total cellular mass upper bound,lifespan in rodents,lifespan in rodents lower bound,lifespan in rodents upper bound,extrapolated cellular turnover rate,extrapolated cellular turnover rate lower bound,extrapolated cellular turnover rate upper bound,extrapolated cellular mass turnover rate,extrapolated cellular mass turnover rate lower bound,extrapolated cellular mass turnover rate upper bound
cell type,,,,,,,,,,,,,,,,,,,,,
Erythrocytes,"25,500,000,000,000.000","25,000,000,000,000.000","26,000,000,000,000.000",119.000,116.000,122.000,"214,000,000,000.000","210,000,000,000.000","220,000,000,000.000",94.000,...,"2,500.000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Neutrophils,"640,000,000,000.000","580,000,000,000.000","700,000,000,000.000",6.600,6.100,7.100,"60,000,000,000.000","45,000,000,000.000","75,000,000,000.000",300.000,...,220.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Monocytes,"5,200,000,000.000","3,500,000,000.000","7,600,000,000.000",3.500,3.000,4.100,"1,520,000,000.000","1,000,000,000.000","2,300,000,000.000",460.000,...,3.600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Thymocytes,"2,300,000,000.000","460,000,000.000","11,500,000,000.000",NaN,NaN,NaN,NaN,NaN,NaN,190.000,...,2.100,3.800,1.650,8.700,"610,000,000.000","100,000,000.000","3,700,000,000.000",0.113,0.019,0.690
Mature T cells,"710,000,000,000.000","320,000,000,000.000","1,560,000,000,000.000",320.000,240.000,430.000,"2,200,000,000.000","960,000,000.000","5,100,000,000.000",190.000,...,290.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
B cells progenitors,"5,500,000,000.000","3,800,000,000.000","7,900,000,000.000",NaN,NaN,NaN,NaN,NaN,NaN,190.000,...,1.490,0.830,0.410,1.660,"6,600,000,000.000","3,700,000,000.000","11,900,000,000.000",1.230,0.680,2.200
Transitional B cells,"2,800,000,000.000","1,170,000,000.000","6,700,000,000.000",NaN,NaN,NaN,NaN,NaN,NaN,190.000,...,1.270,9.900,8.500,11.600,"290,000,000.000","121,000,000.000","700,000,000.000",0.053,0.022,0.127
Mature B cells,"320,000,000,000.000","145,000,000,000.000","700,000,000,000.000",63.000,52.000,76.000,"5,100,000,000.000","2,200,000,000.000","11,700,000,000.000",190.000,...,130.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dermal fibroblasts,"38,000,000,000.000","22,000,000,000.000","54,000,000,000.000",NaN,NaN,NaN,NaN,NaN,NaN,"2,300.000",...,130.000,140.000,120.000,160.000,"270,000,000.000","150,000,000.000","390,000,000.000",0.600,0.300,0.900


In [10]:
pd.read_csv("/Users/jayashankara2/Downloads/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt",sep="\t")

,SAMPID,SMATSSCR,SMCENTER,SMPTHNTS,SMRIN,SMTS,SMTSD,SMUBRID,SMTSISCH,SMTSPAX,...,SME1ANTI,SMSPLTRD,SMBSMMRT,SME1SNSE,SME1PCTS,SMRRNART,SME1MPRT,SMNUM5CD,SMDPMPRT,SME2PCTS
0,GTEX-1117F-0003-SM-58Q7G,NaN,B1,NaN,NaN,Blood,Whole Blood,0013756,"1,188.000",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GTEX-1117F-0003-SM-5DWSB,NaN,B1,NaN,NaN,Blood,Whole Blood,0013756,"1,188.000",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,GTEX-1117F-0003-SM-6WBT7,NaN,B1,NaN,NaN,Blood,Whole Blood,0013756,"1,188.000",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GTEX-1117F-0011-R10a-SM-AHZ7F,NaN,"B1, A1",NaN,NaN,Brain,Brain - Frontal Cortex (BA9),0009834,"1,193.000",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GTEX-1117F-0011-R10b-SM-CYKQ8,NaN,"B1, A1",NaN,7.200,Brain,Brain - Frontal Cortex (BA9),0009834,"1,193.000",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22946,K-562-SM-E9EZC,NaN,NaN,NaN,NaN,Bone Marrow,Cells - Leukemia cell line (CML),EFO_0002067,NaN,NaN,...,"26,289,400.000","27,814,300.000",0.002,"26,121,600.000",49.840,0.006,0.995,NaN,0.000,50.262
22947,K-562-SM-E9EZI,NaN,NaN,NaN,NaN,Bone Marrow,Cells - Leukemia cell line (CML),EFO_0002067,NaN,NaN,...,"26,653,800.000","28,341,700.000",0.002,"26,553,400.000",49.906,0.007,0.995,NaN,0.000,50.205
22948,K-562-SM-E9EZO,NaN,NaN,NaN,NaN,Bone Marrow,Cells - Leukemia cell line (CML),EFO_0002067,NaN,NaN,...,"14,317,500.000","15,168,000.000",0.002,"14,163,500.000",49.730,0.007,0.995,NaN,0.000,50.241
22949,K-562-SM-E9EZT,NaN,NaN,NaN,NaN,Bone Marrow,Cells - Leukemia cell line (CML),EFO_0002067,NaN,NaN,...,"25,459,900.000","26,906,500.000",0.002,"25,259,100.000",49.802,0.007,0.995,NaN,0.000,50.253
